# Diffusion-Augmented MNIST Adversarial Defense Experiment

This notebook is prepared for a final course-project repository. It assumes the project repository contains:

```text
src/adversarial_kd_mnist.py
```

The notebook does **not** clone an external GitHub repository. Instead, it locates the project root by searching upward from the current working directory for `src/adversarial_kd_mnist.py`.

It does the following:

1. Installs the extra dependency `diffusers`.
2. Imports the existing MNIST adversarial-defense code from `src/adversarial_kd_mnist.py`.
3. Trains a class-conditional DDPM-style diffusion model on MNIST.
4. Generates MNIST-like synthetic samples.
5. Adds the synthetic samples to the original MNIST training set.
6. Trains and evaluates:
   - `baseline_diffaug`
   - `adv_training_diffaug`
   - `adv_kd_diffaug`
7. Saves clean, FGSM, and PGD results.

Recommended repository location:

```text
notebooks/diffusion_augmented_mnist_colab.ipynb
```

Recommended result location after running:

```text
results/diffusion_augmented_mnist/
```


In [ ]:
# ============================================================
# 0. Check GPU
# ============================================================

!nvidia-smi || echo "No NVIDIA GPU detected. The notebook can still run on CPU, but it will be much slower."


In [ ]:
# ============================================================
# 1. Install packages and locate project root
# ============================================================

import os
import sys
import json
import zipfile
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

# Install the extra dependency used by this notebook.
# The project itself should also list this in requirements.txt.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "diffusers",
    "accelerate",
])

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from diffusers import UNet2DModel, DDPMScheduler


def find_project_root(start_path=None):
    """Find the repository root containing src/adversarial_kd_mnist.py.

    This avoids cloning another repository inside the notebook.
    Put this notebook inside the final project repo, or run it from the repo root.
    """
    start = Path(start_path or Path.cwd()).resolve()
    candidates = [start, *start.parents]

    # Common Colab location if the user manually cloned the final repo there.
    candidates.append(Path("/content/ML_Project"))

    for candidate in candidates:
        target = candidate / "src" / "adversarial_kd_mnist.py"
        if target.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find src/adversarial_kd_mnist.py.\n"
        "Expected structure:\n"
        "  project_root/src/adversarial_kd_mnist.py\n"
        "  project_root/notebooks/diffusion_augmented_mnist_colab.ipynb\n\n"
        "If running in Colab, first clone or upload the final project repo, "
        "then run this notebook from inside that repo."
    )


REPO_ROOT = find_project_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Project root:", REPO_ROOT)

# Install original repo requirements if available.
requirements_path = REPO_ROOT / "requirements.txt"
if requirements_path.exists():
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_path),
    ])

# Import original project code.
from src.adversarial_kd_mnist import (
    Config,
    EPSILONS,
    SmallCNN,
    set_seed,
    get_device,
    get_loaders,
    accuracy,
    fgsm_attack,
    pgd_attack,
    evaluate_under_attack,
    train_supervised,
    train_adversarial,
    train_adversarial_kd,
    save_checkpoint,
    save_results_csv,
    plot_curves,
)

device = get_device()
print("Using device:", device)


## 2. Experiment settings

The default values are **pilot settings** so the notebook can run on Colab more easily.

For a final run, you can increase:

```python
CLASSIFIER_EPOCHS = 10 or 15
DIFFUSION_EPOCHS = 10
SYNTHETIC_PER_CLASS = 1000 or 2000
PGD_STEPS = 40
```

Large checkpoint files are saved under `outputs/`. For GitHub submission, usually commit the notebook, CSV files, and figures under `results/`, but avoid committing large `.pt` files unless required by the instructor.


In [ ]:
# ============================================================
# 2. Experiment settings
# ============================================================

set_seed(42)

# Pilot setting
CLASSIFIER_EPOCHS = 5
DIFFUSION_EPOCHS = 5
SYNTHETIC_PER_CLASS = 500
NUM_DIFFUSION_STEPS = 200
PGD_STEPS = 40
EVAL_REPEATS = 3

config = Config(
    data_dir="data",
    output_dir="outputs",
    run_name="diffusion_augmented_mnist",
    epochs=CLASSIFIER_EPOCHS,
    batch_size=128,
    lr=1e-3,
    seed=42,
    temperature=5.0,
    kd_alpha=0.5,
    adv_weight=0.5,
    pgd_steps=PGD_STEPS,
    pgd_step_size=0.01,
    adv_train_epsilon=0.1,
    quick_test=False,
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path(config.output_dir) / f"{config.run_name}_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)

# This directory is small and suitable for committing result tables/figures.
repo_results_dir = REPO_ROOT / "results" / "diffusion_augmented_mnist"
repo_results_dir.mkdir(parents=True, exist_ok=True)

print("Output dir:", output_dir)
print("Repo results dir:", repo_results_dir)

with (output_dir / "experiment_config.json").open("w", encoding="utf-8") as f:
    json.dump(
        {
            "CLASSIFIER_EPOCHS": CLASSIFIER_EPOCHS,
            "DIFFUSION_EPOCHS": DIFFUSION_EPOCHS,
            "SYNTHETIC_PER_CLASS": SYNTHETIC_PER_CLASS,
            "NUM_DIFFUSION_STEPS": NUM_DIFFUSION_STEPS,
            "PGD_STEPS": PGD_STEPS,
            "EVAL_REPEATS": EVAL_REPEATS,
            "config": config.__dict__,
        },
        f,
        indent=2,
    )


In [ ]:
# ============================================================
# 3. Load original MNIST train/test loaders
# ============================================================

real_train_loader, test_loader = get_loaders(config)

real_train_dataset = real_train_loader.dataset

print("Real train size:", len(real_train_dataset))
print("Test size:", len(test_loader.dataset))


## 4. Train a conditional DDPM using `diffusers`

The diffusion model is class-conditional because `num_class_embeds=10`, so generated samples can be assigned labels directly.


In [ ]:
# ============================================================
# 4. Define and train diffusion model using diffusers
# ============================================================

noise_scheduler = DDPMScheduler(
    num_train_timesteps=NUM_DIFFUSION_STEPS,
    beta_schedule="linear",
)

diffusion_model = UNet2DModel(
    sample_size=28,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(64, 128, 128),
    down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "UpBlock2D", "UpBlock2D"),
    norm_num_groups=8,
    num_class_embeds=10,   # labels 0-9
).to(device)

optimizer = optim.AdamW(diffusion_model.parameters(), lr=2e-4)


def train_diffusion_model(
    diffusion_model,
    noise_scheduler,
    train_loader,
    optimizer,
    device,
    epochs,
):
    diffusion_model.train()

    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        total_count = 0

        for x, y in tqdm(train_loader, desc=f"Diffusion epoch {epoch}/{epochs}"):
            x = x.to(device)
            y = y.to(device)

            # Original MNIST is [0, 1].
            # Diffusion model trains on [-1, 1].
            clean_images = x * 2.0 - 1.0

            noise = torch.randn_like(clean_images)
            batch_size = clean_images.shape[0]

            timesteps = torch.randint(
                0,
                noise_scheduler.config.num_train_timesteps,
                (batch_size,),
                device=device,
            ).long()

            noisy_images = noise_scheduler.add_noise(
                clean_images,
                noise,
                timesteps,
            )

            noise_pred = diffusion_model(
                noisy_images,
                timesteps,
                class_labels=y,
            ).sample

            loss = F.mse_loss(noise_pred, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_size
            total_count += batch_size

        avg_loss = total_loss / total_count
        print(f"Diffusion epoch {epoch}/{epochs}, loss = {avg_loss:.4f}")


train_diffusion_model(
    diffusion_model=diffusion_model,
    noise_scheduler=noise_scheduler,
    train_loader=real_train_loader,
    optimizer=optimizer,
    device=device,
    epochs=DIFFUSION_EPOCHS,
)

torch.save(
    diffusion_model.state_dict(),
    output_dir / "mnist_conditional_diffusion_model.pt",
)

print("Saved diffusion model to:", output_dir / "mnist_conditional_diffusion_model.pt")


## 5. Generate and save synthetic MNIST

The generated synthetic images are saved to the current run directory as:

```text
outputs/diffusion_augmented_mnist_<timestamp>/synthetic_mnist.pt
```

This file stores:

```python
{
    "images": synthetic_images,
    "labels": synthetic_labels
}
```

Usually this `.pt` file should **not** be committed to GitHub unless the instructor explicitly requires generated data/checkpoints.


In [ ]:
# ============================================================
# 5. Generate synthetic MNIST samples
# ============================================================

@torch.no_grad()
def generate_synthetic_mnist(
    diffusion_model,
    noise_scheduler,
    device,
    num_per_class=500,
    batch_size=128,
    num_inference_steps=200,
):
    diffusion_model.eval()

    all_labels = torch.arange(10).repeat_interleave(num_per_class)

    all_images = []
    all_y = []

    # Set timesteps
    noise_scheduler.set_timesteps(num_inference_steps)

    # Move scheduler tensors to the same device
    noise_scheduler.timesteps = noise_scheduler.timesteps.to(device)

    if hasattr(noise_scheduler, "betas"):
        noise_scheduler.betas = noise_scheduler.betas.to(device)
    if hasattr(noise_scheduler, "alphas"):
        noise_scheduler.alphas = noise_scheduler.alphas.to(device)
    if hasattr(noise_scheduler, "alphas_cumprod"):
        noise_scheduler.alphas_cumprod = noise_scheduler.alphas_cumprod.to(device)

    for start in tqdm(
        range(0, len(all_labels), batch_size),
        desc="Generating synthetic MNIST",
    ):
        labels = all_labels[start:start + batch_size].to(device)
        current_batch_size = labels.shape[0]

        images = torch.randn(
            current_batch_size,
            1,
            28,
            28,
            device=device,
        )

        for t in noise_scheduler.timesteps:
            # Do NOT manually convert t again here.
            # It is already on the correct device.
            noise_pred = diffusion_model(
                images,
                t,
                class_labels=labels,
            ).sample

            images = noise_scheduler.step(
                noise_pred,
                t,
                images,
            ).prev_sample

        # Convert from [-1, 1] back to [0, 1]
        images = torch.clamp((images + 1.0) / 2.0, 0.0, 1.0)

        all_images.append(images.cpu())
        all_y.append(labels.cpu())

    synthetic_images = torch.cat(all_images, dim=0)
    synthetic_labels = torch.cat(all_y, dim=0).long()

    synthetic_dataset = TensorDataset(synthetic_images, synthetic_labels)

    return synthetic_dataset, synthetic_images, synthetic_labels


In [ ]:
synthetic_dataset, synthetic_images, synthetic_labels = generate_synthetic_mnist(
    diffusion_model=diffusion_model,
    noise_scheduler=noise_scheduler,
    device=device,
    num_per_class=SYNTHETIC_PER_CLASS,
    batch_size=128,
    num_inference_steps=NUM_DIFFUSION_STEPS,
)

synthetic_path = output_dir / "synthetic_mnist.pt"
torch.save(
    {
        "images": synthetic_images,
        "labels": synthetic_labels,
    },
    synthetic_path,
)
print("Saved synthetic data to:", synthetic_path)


In [ ]:
def show_synthetic_samples_by_class(synthetic_dataset, num_per_digit=5):
    plt.figure(figsize=(num_per_digit * 1.5, 10))

    plot_idx = 1

    for digit in range(10):
        found = 0

        for i in range(len(synthetic_dataset)):
            x, y = synthetic_dataset[i]

            if y.item() == digit:
                plt.subplot(10, num_per_digit, plot_idx)
                plt.imshow(x.squeeze(), cmap="gray")
                plt.title(str(y.item()))
                plt.axis("off")

                plot_idx += 1
                found += 1

                if found == num_per_digit:
                    break

    plt.tight_layout()
    plt.show()


show_synthetic_samples_by_class(synthetic_dataset, num_per_digit=5)

In [ ]:
# ============================================================
# 6. Visualize synthetic samples
# ============================================================

def show_synthetic_samples(synthetic_dataset, n=20):
    plt.figure(figsize=(10, 4))

    for i in range(n):
        x, y = synthetic_dataset[i]
        plt.subplot(2, n // 2, i + 1)
        plt.imshow(x.squeeze(), cmap="gray")
        plt.title(str(y.item()))
        plt.axis("off")

    plt.tight_layout()
    plt.savefig(output_dir / "synthetic_samples.png", dpi=200)
    plt.show()


show_synthetic_samples(synthetic_dataset, n=20)


In [ ]:
# ============================================================
# 7. Build augmented train_loader
# Convert both real MNIST and synthetic MNIST into TensorDataset
# This avoids int/tensor label mismatch in DataLoader
# ============================================================

from torch.utils.data import DataLoader, TensorDataset

# Collect real MNIST training data into tensors
real_images_list = []
real_labels_list = []

real_collect_loader = DataLoader(
    real_train_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
)

for x, y in real_collect_loader:
    real_images_list.append(x.cpu().float())
    real_labels_list.append(torch.as_tensor(y).cpu().long().view(-1))

real_images = torch.cat(real_images_list, dim=0)
real_labels = torch.cat(real_labels_list, dim=0).long().view(-1)

# Make sure synthetic data also has the correct types
synthetic_images_fixed = synthetic_images.cpu().float()
synthetic_labels_fixed = synthetic_labels.cpu().long().view(-1)

# Optional: shuffle synthetic data
perm = torch.randperm(len(synthetic_images_fixed))
synthetic_images_fixed = synthetic_images_fixed[perm]
synthetic_labels_fixed = synthetic_labels_fixed[perm]

# Combine real + synthetic
augmented_images = torch.cat(
    [real_images, synthetic_images_fixed],
    dim=0,
)

augmented_labels = torch.cat(
    [real_labels, synthetic_labels_fixed],
    dim=0,
).long().view(-1)

augmented_train_dataset = TensorDataset(
    augmented_images,
    augmented_labels,
)

augmented_train_loader = DataLoader(
    augmented_train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Real train size:", len(real_images))
print("Synthetic train size:", len(synthetic_images_fixed))
print("Augmented train size:", len(augmented_train_dataset))

# Test one batch
xb, yb = next(iter(augmented_train_loader))
print("One batch image shape:", xb.shape)
print("One batch label shape:", yb.shape)
print("Label dtype:", yb.dtype)
print("First labels:", yb[:10])


## 8. Train and evaluate diffusion-augmented classifiers

This section trains only the diffusion-augmented versions:

- `baseline_diffaug`
- `adv_training_diffaug`
- `adv_kd_diffaug`

It does **not** rerun the non-diffusion baseline experiment. To compare, use the original experiment results separately.


In [ ]:
# ============================================================
# 8. Repeated attack evaluation helper
# ============================================================

def evaluate_under_attack_repeated(
    model,
    loader,
    device,
    attack_fn,
    epsilons,
    repeats=3,
    base_seed=42,
):
    repeated_results = []

    for r in range(repeats):
        set_seed(base_seed + r)

        result = evaluate_under_attack(
            model=model,
            loader=loader,
            device=device,
            attack_fn=attack_fn,
            epsilons=epsilons,
        )

        repeated_results.append(result)

    averaged_results = {}

    for eps in epsilons:
        values = [result[eps] for result in repeated_results]
        averaged_results[eps] = float(np.mean(values))

    return averaged_results


In [ ]:
# ============================================================
# 9. Train baseline / adv_training / adv_kd with augmented data
# ============================================================

def train_and_evaluate_diffusion_augmented_models(
    augmented_train_loader,
    test_loader,
    config,
    device,
    output_dir,
    eval_repeats=3,
):
    set_seed(config.seed)

    # Teacher is needed for adv_kd.
    teacher = SmallCNN(dropout=0.25).to(device)

    baseline = SmallCNN(dropout=0.25).to(device)
    adv_training = SmallCNN(dropout=0.25).to(device)
    adv_kd = SmallCNN(dropout=0.25).to(device)

    print("\n==============================")
    print("Training teacher on augmented data")
    print("==============================")
    train_supervised(
        teacher,
        augmented_train_loader,
        device,
        config,
        name="teacher_diffaug",
    )
    save_checkpoint(teacher, output_dir, "teacher_diffaug")

    print("\n==============================")
    print("Training baseline on augmented data")
    print("==============================")
    train_supervised(
        baseline,
        augmented_train_loader,
        device,
        config,
        name="baseline_diffaug",
    )
    save_checkpoint(baseline, output_dir, "baseline_diffaug")

    print("\n==============================")
    print("Training FGSM adversarial training model on augmented data")
    print("==============================")
    train_adversarial(
        adv_training,
        augmented_train_loader,
        device,
        config,
    )
    save_checkpoint(adv_training, output_dir, "adv_training_diffaug")

    print("\n==============================")
    print("Training adversarial KD model on augmented data")
    print("==============================")
    train_adversarial_kd(
        adv_kd,
        teacher,
        augmented_train_loader,
        device,
        config,
    )
    save_checkpoint(adv_kd, output_dir, "adv_kd_diffaug")

    models = {
        "baseline_diffaug": baseline,
        "adv_training_diffaug": adv_training,
        "adv_kd_diffaug": adv_kd,
    }

    pgd_fn = lambda m, x, y, eps: pgd_attack(
        m,
        x,
        y,
        epsilon=eps,
        steps=config.pgd_steps,
        step_size=config.pgd_step_size,
    )

    results = {}

    for model_name, model in models.items():
        print("\n==============================")
        print(f"Evaluating {model_name}")
        print("==============================")

        clean_acc = accuracy(model, test_loader, device)
        print(f"{model_name} clean accuracy = {clean_acc:.4f}")

        fgsm_results = evaluate_under_attack_repeated(
            model=model,
            loader=test_loader,
            device=device,
            attack_fn=fgsm_attack,
            epsilons=EPSILONS,
            repeats=eval_repeats,
            base_seed=config.seed + 100,
        )

        pgd_results = evaluate_under_attack_repeated(
            model=model,
            loader=test_loader,
            device=device,
            attack_fn=pgd_fn,
            epsilons=EPSILONS,
            repeats=eval_repeats,
            base_seed=config.seed + 1000,
        )

        results[model_name] = {
            "clean": {0.0: clean_acc},
            "fgsm": fgsm_results,
            "pgd": pgd_results,
        }

    save_results_csv(results, output_dir)
    plot_curves(results, output_dir)

    return results


results_diffaug = train_and_evaluate_diffusion_augmented_models(
    augmented_train_loader=augmented_train_loader,
    test_loader=test_loader,
    config=config,
    device=device,
    output_dir=output_dir,
    eval_repeats=EVAL_REPEATS,
)


In [ ]:
# ============================================================
# 10. Save results as table
# ============================================================

rows = []

for model_name, model_results in results_diffaug.items():
    for attack_name, attack_results in model_results.items():
        for eps, acc in attack_results.items():
            rows.append(
                {
                    "model": model_name,
                    "attack": attack_name,
                    "epsilon": eps,
                    "accuracy": acc,
                }
            )

df_results = pd.DataFrame(rows)
display(df_results)

csv_path = output_dir / "diffusion_augmented_results_table.csv"
df_results.to_csv(csv_path, index=False)

print("Saved result table to:", csv_path)


In [ ]:
# ============================================================
# Plot FGSM / PGD curves for the three diffusion-augmented models
# ============================================================

import matplotlib.pyplot as plt
from pathlib import Path

def plot_attack_curves_three_models(
    results,
    output_dir=None,
    model_order=None,
    rename_map=None,
):
    """
    results format:
    {
        "baseline_diffaug": {
            "clean": {0.0: ...},
            "fgsm": {0.0: ..., 0.01: ..., ...},
            "pgd":  {0.0: ..., 0.01: ..., ...},
        },
        ...
    }
    """

    if model_order is None:
        model_order = [
            "baseline_diffaug",
            "adv_training_diffaug",
            "adv_kd_diffaug",
        ]

    if rename_map is None:
        rename_map = {
            "baseline_diffaug": "baseline",
            "adv_training_diffaug": "adv_training",
            "adv_kd_diffaug": "adv_kd",
        }

    attacks = ["fgsm", "pgd"]

    if output_dir is not None:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    for attack in attacks:
        plt.figure(figsize=(9, 6))

        for model_name in model_order:
            if model_name not in results:
                continue

            attack_results = results[model_name][attack]

            epsilons = sorted(attack_results.keys())
            accuracies = [attack_results[e] for e in epsilons]

            plt.plot(
                epsilons,
                accuracies,
                marker="o",
                linewidth=2.5,
                markersize=8,
                label=rename_map.get(model_name, model_name),
            )

        plt.title(f"MNIST accuracy under {attack.upper()} attack", fontsize=20)
        plt.xlabel("epsilon", fontsize=16)
        plt.ylabel("accuracy", fontsize=16)
        plt.xticks(fontsize=13)
        plt.yticks(fontsize=13)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=14)
        plt.tight_layout()

        if output_dir is not None:
            save_path = output_dir / f"{attack}_three_models.png"
            plt.savefig(save_path, dpi=200)
            print(f"Saved {attack.upper()} plot to: {save_path}")

        plt.show()

In [ ]:
plot_attack_curves_three_models(results_diffaug, output_dir=output_dir)

In [ ]:
# ============================================================
# 11. Zip outputs for download
# ============================================================

zip_path = Path(str(output_dir) + ".zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in output_dir.rglob("*"):
        zf.write(file_path, file_path.relative_to(output_dir.parent))

print("All outputs saved to:", output_dir)
print("Zip file saved to:", zip_path)


In [ ]:
# ============================================================
# 12. Copy small submission-friendly artifacts to results/
# ============================================================

# Copy only lightweight files that are useful for GitHub submission.
# Large .pt checkpoints and generated tensors remain in outputs/.
small_artifacts = [
    "experiment_config.json",
    "synthetic_samples.png",
    "fgsm_three_models.png",
    "pgd_three_models.png",
    "diffusion_augmented_results_table.csv",
]

for name in small_artifacts:
    src_path = output_dir / name
    if src_path.exists():
        shutil.copy2(src_path, repo_results_dir / name)
        print("Copied:", repo_results_dir / name)

# Also save the result table directly in the repo result folder.
if "df_results" in globals():
    df_results.to_csv(repo_results_dir / "diffusion_augmented_results_table.csv", index=False)

print("GitHub-friendly results are in:", repo_results_dir)


## Optional: Save outputs to Google Drive

This section is only for personal Colab backup. It is not needed for the GitHub submission.

If you want to use it, uncomment the code in the next cell.


In [ ]:
# Optional Colab backup. Uncomment only if needed.
# from google.colab import drive
# drive.mount("/content/drive")
#
# drive_output_dir = Path("/content/drive/MyDrive/ML_Project_outputs")
# drive_output_dir.mkdir(parents=True, exist_ok=True)
#
# shutil.copytree(output_dir, drive_output_dir / output_dir.name, dirs_exist_ok=True)
# shutil.copy2(zip_path, drive_output_dir / zip_path.name)
#
# print("Copied outputs to:", drive_output_dir)


## Optional: Load saved synthetic data without retraining diffusion

After running the notebook once, you can reuse `synthetic_mnist.pt` instead of regenerating synthetic data.


In [ ]:
# Optional example:
# saved = torch.load(output_dir / "synthetic_mnist.pt")
# synthetic_images = saved["images"]
# synthetic_labels = saved["labels"]
# synthetic_dataset = TensorDataset(synthetic_images, synthetic_labels)
